# Utilizando modelos e entendendo sua comunicação

Este notebook reúne todos os conceitos e exemplos práticos para o uso de modelos (LLMs) e suas aplicações. Nele você entenderá como a comunicação com um modelo é realizada, como você pode fazer suas próprias perguntas, o que é um *prompt*, os diferentes tipos de prompts, os dados de consumo de um modelo e sua importância para governança. Utilizaremos a plataforma **Groq** para aprendizado, dado a gama de modelos oferecidos de forma gratuita, mas você pode usar a plataforma que melhor lhe agrade ou até modelos *self-hosted*. Também abordaremos dicas que atuarão como um catalisador para o nosso conhecimento.

Pronto? Vamos lá !!! 🚀

## Instalando e importando bibliotecas

Bem ... esse aqui é o primeiro passo, é onde a jornada inicia. Recomendo que você sempre separe uma célula para instalação de dependências e outra célula para importação. Essa separação facilita a manutenção e caso haja uma atualização basta você reexecutar a célula onde o conteúdo foi alterado.

In [1]:
!pip install -q langchain-groq


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


> 💡 **Dica:**
>
> O "-q" aqui é de *quiet*, as vezes pode ser interessante passar esse parâmetro porque algumas instalações de libs/dependências podem ter logs verbosos e isso pode atrapalhar seu estudo.

In [2]:
import os
from langchain_groq import ChatGroq

## Conectando a plataforma Groq para consumo de modelos

Até o momento de escrita deste Jupyter Notebook o site para criação de sua conta na plaforma Groq é https://console.groq.com/home. Acesse e crie sua conta.

In [14]:
os.environ["GROQ_API_KEY"] = "<api-key>"

Precisamos definir a variável de ambiente que contenha sua API Key, que você criou na sua conta Groq. Esse dado é intransferível e privado, tome cuidado ao adicionar esse tipo de informação em repositórios públicos.

Caso não façamos essa definição será necessário informar a API Key em cada chamada ao modelo.

In [15]:
os.getenv("GROQ_API_KEY")

'<api-key>'

> 💡 **Dica:**
>
> Você pode usar qual plataforma quiser, mas a Groq é uma boa opção para aprendizado, dado que oferece uma gama de modelos gratuitos. Caso queira usar outra plataforma, basta alterar o código de conexão e as chamadas aos modelos. Caso queira usar modelos *self-hosted*, você pode utilizar a biblioteca **LangChain** para facilitar a integração e não precisará das configurações de API Key.

## Realizando minha primeira chamada a um modelo

Até o momento de escrita deste Jupyter Notebook a plataforma Groq oferece uma gama de modelos gratuitos, mas você pode utilizar qualquer outro modelo que desejar. Abaixo temos um exemplo de chamada a um modelo da Groq, mas você pode alterar o código para chamar outro modelo ou até mesmo outro provedor. Atenção! Existe um limite de tokens que cada modelo pode consumir na sua conta gratuita, então fique atento a isso. Caso queira consumir mais tokens, você precisará contratar um plano pago.

Vamos desenvolver agora uma função que fará o carregamento do modelo.

In [3]:
def load_llm(id_model: str, temperature: float = 0.7) -> ChatGroq:
    llm = ChatGroq(
        model=id_model,
        temperature=temperature,
        max_tokens = None,
        timeout = None,
        max_retries = 2
    )
    return llm

Esta função recebe um `id_model` que é o identificador do modelo que você deseja utilizar e um parâmetro `temperature` que é um valor entre 0 e 1 que indica a aleatoriedade da resposta do modelo. Quanto mais próximo de 0, mais determinística será a resposta, enquanto valores próximos de 1 tornam a resposta mais criativa e variada.

A classe `ChatGroq` é uma abstração do pacote langchain-groq que facilita a comunicação com os modelos da Groq. Ela encapsula a lógica de conexão e envio de prompts, permitindo que você se concentre na criação de prompts e interpretação das respostas. Existem diversos parâmetros que podem ser passados para a classe `ChatGroq` e você pode customizar a função `load_llm` para receber esses parâmetros e passar para a classe `ChatGroq`.

In [7]:
llm = load_llm(id_model="openai/gpt-oss-20b", temperature=0.7)

In [9]:
answer = llm.invoke(input="Escreva um texto de 3 parágrafos sobre a importância da inteligência artificial na educação.")

In [10]:
answer

AIMessage(content='A inteligência artificial (IA) está transformando a educação ao tornar o processo de aprendizado mais personalizado e eficiente. Por meio de algoritmos que analisam o desempenho de cada estudante, sistemas de recomendação podem sugerir conteúdos, exercícios e estratégias de estudo que se adequam ao seu ritmo e estilo cognitivo. Isso reduz a lacuna entre alunos de diferentes níveis de conhecimento, permitindo que todos avancem no próprio compasso, sem a necessidade de aulas em massa e generalizadas.\n\nAlém da personalização, a IA facilita o acompanhamento em tempo real do progresso dos estudantes. Plataformas educacionais equipadas com análise preditiva conseguem identificar padrões de dificuldade antes mesmo que o aluno perceba, possibilitando intervenções precoces por parte dos professores. Isso não só melhora os resultados acadêmicos, mas também alivia a carga administrativa dos educadores, que passam a dedicar mais tempo ao ensino de habilidades críticas e à inte

Após o modelo ser invocado, o retorno esperado é um objeto do tipo `AIMessage` que contém a resposta do modelo, bem como metadados adicionais sobre a resposta. Você pode acessar o conteúdo da resposta através do atributo `content`, enquanto os metadados adicionais podem ser acessados através dos atributos `additional_kwargs` e `response_metadata`.

In [11]:
answer.content

'A inteligência artificial (IA) está transformando a educação ao tornar o processo de aprendizado mais personalizado e eficiente. Por meio de algoritmos que analisam o desempenho de cada estudante, sistemas de recomendação podem sugerir conteúdos, exercícios e estratégias de estudo que se adequam ao seu ritmo e estilo cognitivo. Isso reduz a lacuna entre alunos de diferentes níveis de conhecimento, permitindo que todos avancem no próprio compasso, sem a necessidade de aulas em massa e generalizadas.\n\nAlém da personalização, a IA facilita o acompanhamento em tempo real do progresso dos estudantes. Plataformas educacionais equipadas com análise preditiva conseguem identificar padrões de dificuldade antes mesmo que o aluno perceba, possibilitando intervenções precoces por parte dos professores. Isso não só melhora os resultados acadêmicos, mas também alivia a carga administrativa dos educadores, que passam a dedicar mais tempo ao ensino de habilidades críticas e à interação humana com o

In [12]:
answer.additional_kwargs

{'reasoning_content': 'The user: "Escreva um texto de 3 parágrafos sobre a importância da inteligência artificial na educação." They want a text in Portuguese, 3 paragraphs. Should be about importance of AI in education. Provide a concise but thorough explanation. Should not exceed maybe 3 paragraphs. Each paragraph maybe 4-6 sentences. Provide a cohesive text. Let\'s produce.'}

In [13]:
answer.response_metadata

{'token_usage': {'completion_tokens': 365,
  'prompt_tokens': 91,
  'total_tokens': 456,
  'completion_time': 0.394607473,
  'completion_tokens_details': {'reasoning_tokens': 77},
  'prompt_time': 0.005645533,
  'prompt_tokens_details': None,
  'queue_time': 0.287850507,
  'total_time': 0.400253006},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_5979a0e1b7',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}